# **LOSS-WEIGHTED TRAINING WARNING**

**THIS NOTEBOOK USES CLASSIFICATION RESULTS TRAINED WITH CLASS-WEIGHTED LOSS INSTEAD OF WEIGHTED WINDOW SAMPLING. ITS MODEL COMPARISONS SHOULD NOT BE INTERPRETED AS RESULTS FROM THE CORRECTED SAMPLING PROCEDURE.**


# Feature–F1 correlation

This notebook asks which physical properties of a fixed-$k$ neighborhood correlate with held-out **window-level F1**. Performance is scored as F1 everywhere, never as the raw share of windows predicted correctly: the window population is dominated by a few populous classes, so raw correctness tracks a bin's class mix as much as it tracks the feature being plotted.

An F1 is a property of a group of windows, not of a single window, so the correlation tables are Spearman rho between a bin's median feature value and that bin's F1, over ~10 quantile bins — read the binned curve first, and rho as a summary of how monotone it is.

Three things vary across the runs here, and every panel holds two of them fixed: the **model type** (the aggregation architecture), the **model ablation** (which of the two input switches — **position** and **LPE** — that architecture gets), and the **node count** (10, 20 or 40 embeddings). Where a panel says "for each model" there are two dropdowns: architecture, then that architecture's ablations, named by the switches alone.

### What "class" means here

**The model only ever predicts the 8-class level-2 hierarchy.** `ACTIVE_HIERARCHY_TREE` is truncated by `HIERARCHY_LEVELS_DROPPED = 2` and pruned by `DROP_LABELS = {'OPC'}`, leaving `neuron`/`non_neuron` → `excitatory`/`inhibitory` → pyramidal, thalamocortical, the three putative interneuron families, and astrocyte / oligo / microglia.

The class dropdown reaches **every class of that tree and every granular label below it**, listed coarse to fine and indented by depth:

- A **coarse** node (`neuron`, `excitatory`) is scored by pooling its descendants' codes. Exact, not approximate — the model emits a level-2 code, so "did the prediction land inside this node" is a question the cached predictions already answer.
- A **granular** label (`L4IT`, `AltBasket`, `NMC`) changes *which windows are scored* and nothing else. The model can never say `AltBasket`, so that group's truth comes from the manifest while its predicted side is still `putative_cge` — **the F1 stays a level-2 F1.** Its false positives include the sibling subtypes under the same level-2 class, which the model has no way to tell apart, so a subtype that is one of eight under `pyramidal` cannot score near 1 however well it is recognised. Compare subtypes against each other, not against the pooled number, and read `accuracy` (section 7) as the "did it get this cell type right" number.

**This notebook never performs inference or feature extraction.** It reads compact analysis summaries from `analysis/all_windows/feature_prediction_cache/`. The batch builder reuses the global per-window prediction cache at `/orcd/scratch/orcd/013/jcbliao/segclr/window_prediction_cache`; missing caches must be built through the Slurm batch script.

**All results on this page are `fold_0`** — one 80/20 split of whole cells at `split_seed` 0. There is no cross-validation in this pipeline, so every score is a single-fold estimate and every margin is a margin on that one split.

In [ ]:
# Auto-reload the analysis modules on every cell run. Without it, editing
# analysis/all_windows/*.py has no effect until the kernel restarts: Python caches modules
# in sys.modules, so re-running this import cell re-imports the OLD object and a
# newly added name fails with ImportError rather than appearing.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'results' / 'all_windows').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from analysis.all_windows.feature_prediction_correlation import (
    CACHE, architecture_of, cached_runs, class_bar_selector, class_options,
    family_selector, feature_selector, has_summary, load_frames, model_of,
    model_order, plot_class_bars, plot_feature_by_model, score_cdf_selector,
    select_runs, stale_summaries, usable_runs)

runs = usable_runs()
print(f'{len(runs)} runs with a best checkpoint, {sum(map(has_summary, runs))} of them cached')
print('models with a cache:', *sorted({model_of(r) for r in runs if has_summary(r)}), sep='\n  ')
print('\nmodels still uncached:',
      *sorted({model_of(r) for r in runs if not has_summary(r)}), sep='\n  ')

## Select runs and load precomputed caches

**The default is every model that has a cache**, one panel per model — that is what makes sections 1 and 2 a grid *of all models* rather than a grid of one run per architecture. The older `select_runs(architectures=...)` collapses each architecture to a single run and picks whichever sorts first by name, which silently answers "which GT?" with a coin toss: the four GT ablations at two depths all carry the architecture `GT`.

**Read the model labels, not the architecture.** A model with neither input switch on is labelled by its bare name — so `GT L4` means *GT depth 4 with rel-pos and LPE both off*, not the default GT, and `MPNN` means MPNN with neither. The ablation dropdowns elsewhere on this page spell the same thing out as `no position, no LPE`.

A run with no cache is listed above with the `sbatch` line that builds it. Narrow the selection by hand when a page of panels is too many:

```python
SELECTED_RUNS = select_runs(runs)                                       # mean vs. pointwise MLP
SELECTED_RUNS = select_runs(runs, models=['MPNN', 'MPNN + LPE'])        # named ablations
SELECTED_RUNS = ['gnn_lcpn_scratch_gt_L4_H4_resnet4x128_n10_fold0']     # a specific run
```

Each uncached run costs one GPU inference pass over every held-out window unless its predictions are already in the shared window-prediction cache, in which case the build is only the geometry join. Do not build a cache on the login node.

In [ ]:
SELECTED_RUNS = cached_runs(runs)
if not SELECTED_RUNS:
    raise FileNotFoundError(
        'No cached runs. Build one with\n'
        '  sbatch scripts/sbatch/build_feature_prediction_cache.sh RUN_NAME')

uncached = [run for run in runs if not has_summary(run)]
if uncached:
    print(f'{len(uncached)} runs have no cache yet; build them from the repo root:')
    for run in uncached:
        print(f'  sbatch scripts/sbatch/build_feature_prediction_cache.sh {run}')
    print()

frames = load_frames(SELECTED_RUNS)
overall_df, corr_df, bins_df = frames['overall'], frames['correlations'], frames['bins']
quantiles_df, cell_recall = frames['quantiles'], frames['cells']
CLASSES = frames['classes']

# A summary written before the granular groups and the quantile marks existed
# still opens: the coarse panels all work off it, and only sections 2-5's
# dotted verticals and section 7 need the newer tables.
stale = stale_summaries(frames)
if stale:
    print('These summary tables predate the current schema:', *stale, sep='\n  ')
    print('Rebuild them (no inference, no GPU) with:\n'
          '  sbatch scripts/sbatch/summarize_feature_prediction_cache.sh\n')

print(f'Loaded compact summaries for {len(SELECTED_RUNS)} runs; no raw windows loaded')
print('models on the grid:', *model_order(bins_df), sep='\n  ')
print('scored classes, coarse to fine:', *class_options(bins_df, CLASSES), sep='\n  ')
display(overall_df[overall_df.class_name.isna()].drop(
    columns=['class_name', 'coarse_class', 'granular'], errors='ignore'))

## 1. Path length

`path_distance_um` is the total skeleton cable in the induced neighborhood — how much of the cell the $k$ embeddings actually span, which varies with node spacing even though $k$ is fixed.

One panel per model, one line per node count. Three curves converging inside a panel means the feature stops mattering once the neighborhood is large; one panel disagreeing with its neighbours means that aggregation is uniquely sensitive to it.

In [ ]:
feature_selector(bins_df, 'path_distance_um', CLASSES)

## 2. Node density

`node_density_per_um` is $k$ divided by that cable length: how tightly the same fixed number of embeddings is packed along the process. It is the reciprocal view of section 1, and the two are worth reading together — a model tracking cable but not density is responding to extent, and the reverse is responding to sampling.

**Dotted verticals** drop from each curve to the axis at the 25th, 50th and 75th percentile *of that curve's own feature distribution*, in the curve's colour. They say where the mass of that series actually sits, so a dramatic swing to the right of p75 is a swing over the last quarter of windows and should be read as such. The quantiles come from the raw windows, not from the bin edges — ten quantile bins put edges at the deciles, and p25 is not one of them.

In [ ]:
feature_selector(bins_df, 'node_density_per_um', CLASSES, quantiles=quantiles_df)

## 3. New-skeleton clearance radius

Each embedding node is joined by nearest coordinate to the **new** skeleton at `/orcd/scratch/orcd/013/jcbliao/skeletons/segclr/skeletons`, and that skeleton's reported `radius` is used. These are not CAVE skeleton radii, and they are not the spine-corrected ray-cast shaft radius under `data/dendrite_thickness_cache/`.

Pick a **model type** and an **ablation**. Mean and median over the $k$ members share a panel, and the panels run left to right across **node count**. The minimum and maximum are still cached but not drawn: they are order statistics of a $k$-sized sample, so they move with $k$ itself rather than with the neighborhood's clearance.

In [ ]:
family_selector(bins_df, 'clearance radius (nm)', CLASSES, quantiles=quantiles_df)

## 4. Segmentation volume contained in the embeddings

Per-node occupancy comes from `data/mask_volume_cache`: the saved count of cell-mask voxels inside the exact box that node's embedding was computed from. Mean and median over the $k$ members share a panel, across node count. The sum is cached but not drawn — it grows with $k$ by construction and double-counts overlapping physical boxes.

In [ ]:
family_selector(bins_df, 'embedding-box volume (um^3)', CLASSES, quantiles=quantiles_df)

## 5. Distance of the center node from the soma

Spatial distance is center-to-nucleus Euclidean distance; path distance is the graph geodesic from the graph node nearest the nucleus to the center node. Disconnected components stay missing rather than being given a fabricated distance.

The two ride on one panel because they measure the same thing two ways, and in practice they agree closely — the panel is where that agreement is checked rather than assumed. This is where the class dropdown matters most: an interneuron's dense local axon and a pyramidal cell's apical trunk sit at very different distances from their somata, so a pooled curve mixes two different geometries.

In [ ]:
family_selector(bins_df, 'distance from soma (um)', CLASSES, quantiles=quantiles_df)

## 6. Spearman rho over the bins

The monotonicity summary for everything plotted above: rho between a bin's median feature value and that bin's F1, over ~10 quantile bins. Ten points, not millions — read it as a description of the curve's shape, not as a test carrying the per-window sample size.

In [ ]:
display(corr_df[corr_df.class_name.isna()].sort_values(
    ['feature', 'model', 'n_embeddings'])[
    ['model', 'n_embeddings', 'feature', 'spearman_rho', 'p', 'n_bins', 'n_windows']])

## 7. Window F1 per granular cell type

One model, every granular label on the x axis, **three adjacent bars per class for the three node counts**. Labels are grouped under the level-2 class the model actually emits for them, with a divider and a heading between groups.

**One panel, because there is one thing being measured: the level-2 prediction.** A bar is the F1 of "this window was given the right level-2 class", scored over that subtype's windows alone. Inside the group every window has the same correct answer, so no window there can be a false positive — precision is 1, and the bar depends on nothing but how often that subtype is called correctly. No sibling subtype enters it.

Scoring a subtype against the whole population instead would put every *correctly*-classified sibling into its false-positive count: `L5ET` then reads F1 0.067 while being called correctly 80.5% of the time, because the number has become L5ET's share of the pyramidal population rather than anything about the model. That is not what these bars show.

In [ ]:
class_bar_selector(overall_df, CLASSES)

## 8. Cumulative distribution of per-cell F1

The distribution is over **held-out cells**, one point each; the score is that cell's F1 for its own class, over its own windows. Inside a cell there is no window that could be a false positive — every window carries that cell's single label — so precision is 1 wherever the class is predicted at all and the F1 is $2r/(1+r)$. It is therefore monotone in that cell's recall: the curves rank cells exactly as a recall CDF would, and what changes is the axis the 0.5 line is read against. The run's **window** macro F1 is annotated in the pooled panel's legend, which is a different quantity — one number for the whole model, not a point on the curve.

The large panel pools every held-out cell; the grid below splits by granular `cell_type`. Three colours are the three node counts, and the dashed vertical line at 0.5 reads off as *this fraction of held-out cells score below F1 0.5*.

In [ ]:
display(cell_recall.groupby(['model', 'n_embeddings']).f1.describe(
    percentiles=[.05, .1, .25, .5, .75, .9, .95]))
score_cdf_selector(cell_recall, overall_df)

## 9. Per-cell table

Every held-out cell with its granular `cell_type`, the level-2 `class_name` the model actually predicts, its F1 and the recall behind it.

In [ ]:
display(cell_recall.sort_values(['model', 'n_embeddings', 'f1'])[
    ['model', 'n_embeddings', 'root_id', 'cell_type', 'class_name', 'f1', 'recall', 'n_windows']])